In [1]:
"""
PASO 3 - Minority Game con abstencion: efecto del numero de agentes N
=====================================================================
Tesis: Complejidad e informacion en un minority game con opcion de abstencion
Base: R. Mansilla, Phys. Rev. E 62, 4553 (2000), Ec. (6)

Alfabeto: 0 = vender, 1 = comprar, 2 = abstenerse

Que hace este script:
  1. Para cada valor de N corre 100 partidas (las mismas 100 semillas
     para todos los N), con agentes de s = 2 estrategias.
  2. A cada cadena de ganadores le aplica M(d), Ec. (6) del PRE 62,
     para d = 1, ..., 100.
  3. Promedia M(d) sobre las 100 cadenas y calcula el intervalo de
     confianza del 95%.
  4. Grafica los N juntos, en grupos de hasta 4 curvas por grafica (se
     arman solos a partir de LISTA_N). Por grupo salen dos graficas: los
     promedios, y los promedios con sus bandas de confianza.
  5. Guarda CSV por cada N y un CSV de resumen con todos los N.

PARA CAMBIAR LOS N: solo se modifica la linea LISTA_N (seccion PARAMETROS).
Los nombres de las graficas y del resumen llevan los N corridos, asi que
no se sobrescriben los archivos de corridas anteriores con otros N.

gamma = 3^m / N es el parametro de control (antes se llamaba alfa).

Parametros fijos: T = 6000, s = 2, m = 3, K = 3, sin burn-in, 100 semillas.
Las funciones de simulacion, de ganador y de M(d) son las mismas del
Segundo paso, sin cambios.

PUNTAJE = True  -> agentes con 2 estrategias CON puntaje (codigo base)
PUNTAJE = False -> agentes con 2 estrategias SIN puntaje
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

try:
    from scipy.stats import t as t_student
    T_95 = None          # se calcula con scipy mas abajo
except ImportError:
    t_student = None
    T_95 = 1.984         # t de Student, 95%, 99 grados de libertad


# =====================================================================
# PARAMETROS
# =====================================================================
SEMILLA_MAESTRA = 1907443528   # genera las 100 semillas de forma reproducible
N_CADENAS = 100                # numero de cadenas (semillas distintas)

M       = 3      # memoria (brain size)
S       = 2      # estrategias por agente
K       = 3      # tamano del alfabeto
T       = 6000   # ciclos registrados, desde el primero (sin burn-in)

PUNTAJE = True   # True = con puntaje, False = sin puntaje

# =====================================================================
# NUMERO DE AGENTES  <-- ESTA ES LA UNICA LINEA QUE HAY QUE CAMBIAR
# =====================================================================
LISTA_N = [125, 151, 175]

# Los grupos de graficas se arman solos: hasta 4 valores de N por grafica.
N_POR_GRAFICA = 4
GRUPOS = [
    ("N" + "-".join(str(n) for n in LISTA_N[i:i + N_POR_GRAFICA]),
     LISTA_N[i:i + N_POR_GRAFICA])
    for i in range(0, len(LISTA_N), N_POR_GRAFICA)
]

D_MAX   = 100    # M(d) se calcula para d = 1, ..., D_MAX
NIVEL   = 0.95   # nivel de confianza del intervalo

ETIQUETA = "con_puntaje" if PUNTAJE else "sin_puntaje"
DESCRIPCION = ("2 estrategias, con puntaje" if PUNTAJE
               else "2 estrategias, sin puntaje")


# =====================================================================
# REGLA DE GANADOR  (identica a la del codigo base)
# =====================================================================
def regla_ganador(conteos, rng):
    """Gana el argmin sobre las K acciones, con desempate aleatorio."""
    empatados = np.flatnonzero(conteos == conteos.min())
    return int(rng.choice(empatados))


# =====================================================================
# s ESTRATEGIAS POR AGENTE, SIN PUNTAJE  (igual que en el Segundo paso)
# =====================================================================
def simular_sin_recompensa(N, m, s, K, T, semilla):
    """Agentes con s estrategias que no las califican: los puntajes
    valen 0 siempre, asi que en cada ciclo cada agente toma al azar una
    de sus estrategias y hace lo que dice para la historia actual.
    """
    rng = np.random.default_rng(semilla)
    N_HIST = K ** m

    # estrategias[i, j, h] = accion que la estrategia j del agente i
    #                        recomienda ante la historia con indice h
    estrategias = rng.integers(0, K, size=(N, s, N_HIST))

    # puntajes[i, j]: se quedan en 0 porque no hay recompensa
    puntajes = np.zeros((N, s))

    # memoria: los m datos que el mercado tiene "al nacer"
    historia = rng.integers(0, K, size=m)

    pesos = K ** np.arange(m - 1, -1, -1)   # convierte historia -> indice
    indices = np.arange(N)

    serie_ganadores = np.zeros(T, dtype=int)

    for t in range(T):

        # 1. que situacion estan viendo los agentes
        idx = int(historia @ pesos)

        # 2. cada agente toma su estrategia con mas puntos.
        #    Como todos los puntajes valen 0, el ruido decide al azar.
        mejor = np.argmax(puntajes + rng.random((N, s)) * 0.5, axis=1)

        # 3. consulta esa estrategia en la historia actual y actua
        acciones = estrategias[indices, mejor, idx]

        # 4. se cuenta cuantos eligieron cada opcion
        conteos = np.bincount(acciones, minlength=K)

        # 5. gana la minoria
        ganador = regla_ganador(conteos, rng)

        # 6. (SIN RECOMPENSA: en el codigo base aqui se sumaban puntos)

        # 7. registrar desde el primer ciclo
        serie_ganadores[t] = ganador

        # 8. la memoria se desplaza: entra el ganador, sale el mas viejo
        historia = np.roll(historia, -1)
        historia[-1] = ganador

    return serie_ganadores


# =====================================================================
# s ESTRATEGIAS POR AGENTE, CON PUNTAJE  (igual que en el Segundo paso)
# =====================================================================
def simular_con_puntaje(N, m, s, K, T, semilla):
    """Reglas del codigo base, sin burn-in. Cada agente usa la estrategia
    con mas puntos y, al final del ciclo, todas las estrategias que
    habrian acertado al ganador reciben un punto.

    Devuelve la serie de ganadores y cuantas veces acerto cada agente.
    """
    rng = np.random.default_rng(semilla)
    N_HIST = K ** m

    # estrategias[i, j, h] = accion que la estrategia j del agente i
    #                        recomienda ante la historia con indice h
    estrategias = rng.integers(0, K, size=(N, s, N_HIST))

    # puntajes[i, j] = puntos virtuales (casos de exito) de esa estrategia
    puntajes = np.zeros((N, s))

    # memoria: los m datos que el mercado tiene "al nacer"
    historia = rng.integers(0, K, size=m)

    pesos = K ** np.arange(m - 1, -1, -1)   # convierte historia -> indice
    indices = np.arange(N)

    serie_ganadores = np.zeros(T, dtype=int)
    exitos_agente = np.zeros(N, dtype=int)

    for t in range(T):

        # 1. que situacion estan viendo los agentes
        idx = int(historia @ pesos)

        # 2. cada agente toma su estrategia con mas puntos.
        #    El ruido < 1 rompe empates al azar sin alterar el orden
        #    entre puntajes enteros distintos.
        mejor = np.argmax(puntajes + rng.random((N, s)) * 0.5, axis=1)

        # 3. consulta esa estrategia en la historia actual y actua
        acciones = estrategias[indices, mejor, idx]

        # 4. se cuenta cuantos eligieron cada opcion
        conteos = np.bincount(acciones, minlength=K)

        # 5. gana la minoria
        ganador = regla_ganador(conteos, rng)

        # 6. se premian TODAS las estrategias que hubieran acertado,
        #    se hayan jugado o no (regla del codigo base / PRE 62).
        puntajes += (estrategias[:, :, idx] == ganador)

        # 7. registrar desde el primer ciclo
        serie_ganadores[t] = ganador
        exitos_agente += (acciones == ganador)

        # 8. la memoria se desplaza: entra el ganador, sale el mas viejo
        historia = np.roll(historia, -1)
        historia[-1] = ganador

    return serie_ganadores, exitos_agente


# =====================================================================
# FUNCION DE INFORMACION MUTUA  -  Ec. (6) del PRE 62
# =====================================================================
def informacion_mutua(serie, d, K):
    """M(d) = sum_{a,b} P_ab(d) * log2[ P_ab(d) / (P_a * P_b) ]

    P_ab(d): probabilidad de encontrar el simbolo a y, d lugares despues,
             el simbolo b. Se estima contando todos los pares
             (serie[i], serie[i+d]) de la cadena.
    P_a:     densidad del simbolo a en la cadena completa (como dice
             Mansilla: "the density of the symbol alpha").

    La suma corre sobre los K*K = 9 pares (a, b). Los pares que nunca
    aparecen (P_ab = 0) no aportan, porque 0 * log(0) = 0.
    """
    x = serie[:-d]          # simbolo alpha
    y = serie[d:]           # simbolo beta, d lugares despues
    n_pares = len(x)

    # probabilidades conjuntas P_ab(d): matriz K x K
    conjunta = np.bincount(x * K + y, minlength=K * K).reshape(K, K) / n_pares

    # densidades P_a de cada simbolo en toda la cadena
    densidad = np.bincount(serie, minlength=K) / len(serie)
    producto = np.outer(densidad, densidad)      # P_a * P_b

    M_d = 0.0
    for a in range(K):
        for b in range(K):
            if conjunta[a, b] > 0:
                M_d += conjunta[a, b] * np.log2(conjunta[a, b] / producto[a, b])
    return M_d


def sesgo_esperado(K, n_pares):
    """Valor que da M(d) en una cadena SIN correlacion, solo por ser finita.

    Es aproximadamente (K-1)^2 / (2 * n * ln 2). Solo depende de T, asi
    que es la misma linea de referencia para todos los N.
    """
    return (K - 1) ** 2 / (2 * n_pares * np.log(2))


def exito_al_azar(N, K, n_muestras=200000, semilla=0):
    """Tasa de exito que tendrian N agentes eligiendo al azar.

    El 0.265 que imprimia el Segundo paso solo vale para N = 50. Aqui se
    calcula para cada N: se reparten N agentes al azar entre las K
    acciones muchas veces y se promedia la fraccion que queda en la
    accion ganadora (la de menos agentes).
    """
    rng = np.random.default_rng(semilla)
    conteos = rng.multinomial(N, [1 / K] * K, size=n_muestras)
    return (conteos.min(axis=1) / N).mean()


# =====================================================================
# CORRER LAS 100 CADENAS PARA UN VALOR DE N
# =====================================================================
def correr_N(N, semillas, distancias):
    tabla = np.zeros((N_CADENAS, D_MAX))     # tabla[c, j] = M(d=j+1)
    frecuencias = np.zeros((N_CADENAS, K))
    tasa_exito = np.zeros(N_CADENAS)

    for c, semilla in enumerate(semillas):
        if PUNTAJE:
            serie, exitos = simular_con_puntaje(N, M, S, K, T, int(semilla))
            tasa_exito[c] = exitos.sum() / (N * T)
        else:
            serie = simular_sin_recompensa(N, M, S, K, T, int(semilla))
            tasa_exito[c] = np.nan
        frecuencias[c] = np.bincount(serie, minlength=K) / T
        for j, d in enumerate(distancias):
            tabla[c, j] = informacion_mutua(serie, d, K)
        if (c + 1) % 10 == 0:
            print(f"  cadenas terminadas: {c + 1}/{N_CADENAS}", flush=True)

    return tabla, frecuencias, tasa_exito


# =====================================================================
# GRAFICAS DE UN GRUPO (todos sus N en la misma grafica)
# =====================================================================
COLORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # uno por cada N
GRIS = "#52514e"
TINTA = "#0b0b0b"


def graficar_grupo(nombre, lista_N, resultados, distancias, referencia):
    plt.rcParams.update({
        "font.size": 11,
        "axes.edgecolor": GRIS,
        "axes.labelcolor": TINTA,
        "xtick.color": GRIS,
        "ytick.color": GRIS,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })
    subtitulo = (f"m={M}, s={S}, K={K}, T={T}, sin burn-in, "
                 f"{N_CADENAS} semillas por cada N")

    # --- Grafica 1: promedios de M(d) para cada N ---
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for N, color in zip(lista_N, COLORES):
        media = resultados[N]["media"]
        ax.plot(distancias, media, color=color, lw=1.8, marker="o", ms=3,
                label=f"N = {N}")
    ax.plot(distancias, referencia, color=GRIS, lw=1.5, ls="--",
            label="Sesgo esperado de una cadena sin correlación")
    ax.set_xlabel("Distancia d")
    ax.set_ylabel("M(d)  [bits]")
    ax.set_title(f"Información mutua promedio — {DESCRIPCION}, "
                 f"N = {', '.join(str(n) for n in lista_N)}\n" + subtitulo,
                 loc="left", color=TINTA)
    ax.set_xlim(0, D_MAX + 1)
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", color="#e5e4df", lw=0.8)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.13),
              ncol=3, fontsize=9)
    fig.tight_layout()
    archivo = f"Paso3_grafica_Md_{ETIQUETA}_{nombre}_promedio.png"
    fig.savefig(archivo, dpi=200)
    plt.close(fig)
    print(f"  Guardado: {archivo}", flush=True)

    # --- Grafica 2: promedios con bandas de confianza ---
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for N, color in zip(lista_N, COLORES):
        r = resultados[N]
        ax.fill_between(distancias, r["ic_inf"], r["ic_sup"], color=color,
                        alpha=0.18, lw=0)
        ax.plot(distancias, r["media"], color=color, lw=1.8,
                label=f"N = {N}")
    ax.plot(distancias, referencia, color=GRIS, lw=1.5, ls="--",
            label="Sesgo esperado de una cadena sin correlación")
    manijas, etiquetas = ax.get_legend_handles_labels()
    manijas.append(Patch(color=GRIS, alpha=0.25, lw=0))
    etiquetas.append(f"Bandas: intervalo de confianza {int(NIVEL*100)}%")
    ax.set_xlabel("Distancia d")
    ax.set_ylabel("M(d)  [bits]")
    ax.set_title(f"Promedio de M(d) con intervalo de confianza del "
                 f"{int(NIVEL*100)}% — {DESCRIPCION}\n"
                 f"N = {', '.join(str(n) for n in lista_N)};  "
                 f"t de Student con {N_CADENAS - 1} grados de libertad",
                 loc="left", color=TINTA)
    ax.set_xlim(0, D_MAX + 1)
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", color="#e5e4df", lw=0.8)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.legend(manijas, etiquetas, frameon=False, loc="upper center",
              bbox_to_anchor=(0.5, -0.13), ncol=3, fontsize=9)
    fig.tight_layout()
    archivo = f"Paso3_grafica_Md_{ETIQUETA}_{nombre}_intervalo_confianza.png"
    fig.savefig(archivo, dpi=200)
    plt.close(fig)
    print(f"  Guardado: {archivo}", flush=True)


# =====================================================================
# PROGRAMA PRINCIPAL
# =====================================================================
if __name__ == "__main__":

    # --- las 100 semillas (las mismas para todos los N) ---
    semillas = np.random.SeedSequence(SEMILLA_MAESTRA).generate_state(N_CADENAS)
    distancias = np.arange(1, D_MAX + 1)

    # --- valor critico de la t de Student ---
    t_crit = (t_student.ppf(0.5 + NIVEL / 2, N_CADENAS - 1)
              if t_student is not None else T_95)

    # --- linea de referencia: igual para todos los N (solo depende de T) ---
    referencia = sesgo_esperado(K, T - distancias)

    resumen = []

    for nombre_grupo, lista_N in GRUPOS:
        resultados = {}

        for N in lista_N:
            gamma = K ** M / N
            print("\n" + "=" * 62)
            print(f"  {DESCRIPCION}  |  {nombre_grupo}")
            print(f"  N={N}  m={M}  s={S}  K={K}  T={T}  burn_in=0"
                  f"  gamma={gamma:.4f}")
            print(f"  cadenas={N_CADENAS}  d = 1..{D_MAX}")
            print("=" * 62, flush=True)

            tabla, frecuencias, tasa_exito = correr_N(N, semillas, distancias)

            # --- promedio sobre las 100 cadenas, para cada d ---
            media = tabla.mean(axis=0)
            desv = tabla.std(axis=0, ddof=1)
            error_est = desv / np.sqrt(N_CADENAS)
            ic_inf = media - t_crit * error_est
            ic_sup = media + t_crit * error_est
            resultados[N] = {"media": media, "ic_inf": ic_inf, "ic_sup": ic_sup}

            tasa_media = tasa_exito.mean() if PUNTAJE else np.nan
            azar = exito_al_azar(N, K)
            d_max = int(distancias[np.argmax(media)])

            # --- resumen en pantalla ---
            print("\n  Frecuencia media de cada simbolo (100 cadenas):")
            for a, nombre in enumerate(["vender", "comprar", "abstenerse"]):
                print(f"    {a} ({nombre:10s}): {frecuencias[:, a].mean():.4f}")
            if PUNTAJE:
                print(f"\n  Tasa de exito media de los agentes: {tasa_media:.4f}"
                      f"  (agentes al azar con N={N}: {azar:.4f})")
            print(f"\n  M(d) promedio: min={media.min():.3e}  "
                  f"max={media.max():.3e} (en d={d_max})  "
                  f"media global={media.mean():.3e} bits")
            print(f"  M(1)={media[0]:.3e}  M(2)={media[1]:.3e}  "
                  f"M(3)={media[2]:.3e}  M(4)={media[3]:.3e}")
            print(f"  Sesgo esperado de una cadena sin correlacion: "
                  f"{referencia.mean():.3e} bits", flush=True)

            # --- guardar resultados de este N ---
            parametros = (f"N={N} m={M} s={S} K={K} T={T} burn_in=0 "
                          f"gamma={gamma:.6f} {ETIQUETA}")
            encabezado = (
                f"PASO 3. M(d) con {DESCRIPCION}, Ec. (6) PRE 62\n"
                f"{parametros} cadenas={N_CADENAS} "
                f"semilla_maestra={SEMILLA_MAESTRA}\n"
                f"intervalo de confianza {int(NIVEL*100)}% con t de Student "
                f"(t={t_crit:.4f})\n"
                f"d,media,desv_estandar,error_estandar,ic_inferior,"
                f"ic_superior,sesgo_esperado"
            )
            np.savetxt(f"Paso3_Md_{ETIQUETA}_N{N}_promedio.csv",
                       np.column_stack([distancias, media, desv, error_est,
                                        ic_inf, ic_sup, referencia]),
                       fmt=["%d"] + ["%.8e"] * 6, delimiter=",",
                       header=encabezado, comments="# ")

            encabezado_todas = (
                f"PASO 3. M(d) de cada cadena. Fila = una semilla, "
                f"columna = d\n{parametros}\n"
                f"semilla," + ",".join(f"d{d}" for d in distancias)
            )
            np.savetxt(f"Paso3_Md_{ETIQUETA}_N{N}_todas_las_cadenas.csv",
                       np.column_stack([semillas.astype(np.int64), tabla]),
                       fmt=["%d"] + ["%.8e"] * D_MAX, delimiter=",",
                       header=encabezado_todas, comments="# ")

            print(f"\n  Guardado: Paso3_Md_{ETIQUETA}_N{N}_promedio.csv")
            print(f"  Guardado: Paso3_Md_{ETIQUETA}_N{N}_todas_las_cadenas.csv",
                  flush=True)

            resumen.append([N, gamma, tasa_media, azar,
                            media[0], media[1], media[2], media[3],
                            media.max(), d_max])

        # --- las graficas del grupo, con todos sus N juntos ---
        print(f"\n  Graficas del {nombre_grupo}:")
        graficar_grupo(nombre_grupo, lista_N, resultados, distancias,
                       referencia)

    # --- resumen de todos los N en un solo archivo ---
    nombre_resumen = (f"Paso3_resumen_{ETIQUETA}_N"
                      + "-".join(str(n) for n in LISTA_N) + ".csv")
    np.savetxt(nombre_resumen, np.array(resumen),
               fmt=["%d", "%.6f", "%.6f", "%.6f",
                    "%.8e", "%.8e", "%.8e", "%.8e", "%.8e", "%d"],
               delimiter=",",
               header=(f"PASO 3. Resumen, {DESCRIPCION}, m={M} s={S} K={K} "
                       f"T={T} cadenas={N_CADENAS}\n"
                       f"N,gamma,tasa_exito,exito_al_azar,"
                       f"M1,M2,M3,M4,M_max,d_del_maximo"),
               comments="# ")
    print(f"\n  Guardado: {nombre_resumen}")
    print("\n  Listo.")


  2 estrategias, con puntaje  |  N125-151-175
  N=125  m=3  s=2  K=3  T=6000  burn_in=0  gamma=0.2160
  cadenas=100  d = 1..100
  cadenas terminadas: 10/100
  cadenas terminadas: 20/100
  cadenas terminadas: 30/100
  cadenas terminadas: 40/100
  cadenas terminadas: 50/100
  cadenas terminadas: 60/100
  cadenas terminadas: 70/100
  cadenas terminadas: 80/100
  cadenas terminadas: 90/100
  cadenas terminadas: 100/100

  Frecuencia media de cada simbolo (100 cadenas):
    0 (vender    ): 0.3347
    1 (comprar   ): 0.3320
    2 (abstenerse): 0.3333

  Tasa de exito media de los agentes: 0.3130  (agentes al azar con N=125: 0.2901)

  M(d) promedio: min=4.151e-04  max=5.556e-03 (en d=3)  media global=6.848e-04 bits
  M(1)=4.820e-03  M(2)=5.449e-03  M(3)=5.556e-03  M(4)=2.861e-03
  Sesgo esperado de una cadena sin correlacion: 4.850e-04 bits

  Guardado: Paso3_Md_con_puntaje_N125_promedio.csv
  Guardado: Paso3_Md_con_puntaje_N125_todas_las_cadenas.csv

  2 estrategias, con puntaje  |  N125-1